# Graph Structures Lab



```{contents}
:local:
:depth: 2
```


This lab builds a small graph representation for a campus-route dataset. The goal is to practice modeling vertices and edges, then choose operations that make later graph algorithms easier to write.


```{index} graph lab
```

## Build the Route Data

Start with vertices and weighted undirected edges. Each edge stores two locations and an estimated walking time in minutes.


In [ ]:
using System;

string[] locations = { "Library", "Dining", "Gym", "Lab", "Dorm" };
(string A, string B, int Minutes)[] paths = {
    ("Library", "Dining", 4),
    ("Library", "Lab", 6),
    ("Dining", "Gym", 5),
    ("Gym", "Dorm", 7),
    ("Lab", "Dorm", 8)
};

Console.WriteLine($"Locations: {locations.Length}");
Console.WriteLine($"Paths: {paths.Length}");


```{index} adjacency list; weighted graph
```

## Build a Weighted Adjacency List

A weighted adjacency list stores each neighbor with the edge weight. Tuples keep the example compact.


In [ ]:
using System;
using System.Collections.Generic;

(string A, string B, int Minutes)[] paths = {
    ("Library", "Dining", 4),
    ("Library", "Lab", 6),
    ("Dining", "Gym", 5),
    ("Gym", "Dorm", 7),
    ("Lab", "Dorm", 8)
};

var graph = new Dictionary<string, List<(string Neighbor, int Minutes)>>();

foreach ((string a, string b, int minutes) in paths)
{
    AddPath(a, b, minutes);
}

foreach ((string place, List<(string Neighbor, int Minutes)> neighbors) in graph)
{
    string summary = string.Join(", ", neighbors.Select(n => $"{n.Neighbor} ({n.Minutes})"));
    Console.WriteLine($"{place}: {summary}");
}

void AddPath(string a, string b, int minutes)
{
    graph.TryAdd(a, new List<(string, int)>());
    graph.TryAdd(b, new List<(string, int)>());
    graph[a].Add((b, minutes));
    graph[b].Add((a, minutes));
}


This representation is ready for algorithms that loop through neighbors, such as breadth-first search, depth-first search, and shortest-path algorithms.


```{index} graph testing
```

## Add Basic Queries

Small helper functions make the representation easier to test. Start with direct-neighbor lookup and degree.


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new Dictionary<string, List<(string Neighbor, int Minutes)>>
{
    ["Library"] = new() { ("Dining", 4), ("Lab", 6) },
    ["Dining"] = new() { ("Library", 4), ("Gym", 5) },
    ["Gym"] = new() { ("Dining", 5), ("Dorm", 7) },
    ["Lab"] = new() { ("Library", 6), ("Dorm", 8) },
    ["Dorm"] = new() { ("Gym", 7), ("Lab", 8) }
};

Console.WriteLine($"Library degree: {Degree("Library")}");
Console.WriteLine($"Dining to Dorm direct? {HasPath("Dining", "Dorm")}");
Console.WriteLine($"Gym to Dorm direct? {HasPath("Gym", "Dorm")}");

int Degree(string vertex)
{
    return graph.TryGetValue(vertex, out var neighbors) ? neighbors.Count : 0;
}

bool HasPath(string from, string to)
{
    return graph.TryGetValue(from, out var neighbors)
        && neighbors.Any(edge => edge.Neighbor == to);
}


These are not full traversal algorithms yet. They are representation checks: can the structure answer basic questions correctly?


```{index} adjacency matrix
```

## Compare With a Matrix

Build the same route data as a weighted matrix. This makes direct edge lookup fast, but it needs a row and column for every pair of locations.


In [ ]:
using System;
using System.Collections.Generic;

string[] locations = { "Library", "Dining", "Gym", "Lab", "Dorm" };
var index = new Dictionary<string, int>();

for (int i = 0; i < locations.Length; i++)
{
    index[locations[i]] = i;
}

int?[,] minutes = new int?[locations.Length, locations.Length];
AddPath("Library", "Dining", 4);
AddPath("Library", "Lab", 6);
AddPath("Dining", "Gym", 5);
AddPath("Gym", "Dorm", 7);
AddPath("Lab", "Dorm", 8);

Console.WriteLine($"Library to Lab: {minutes[index["Library"], index["Lab"]]} minutes");
Console.WriteLine($"Dining to Dorm: {minutes[index["Dining"], index["Dorm"]]?.ToString() ?? "no direct path"}");

void AddPath(string a, string b, int value)
{
    minutes[index[a], index[b]] = value;
    minutes[index[b], index[a]] = value;
}


The matrix version is simple to query once vertex indexes exist. The adjacency-list version is usually simpler to grow, inspect, and traverse.


```{index} graph design
```

## Design Check

Before using a graph in a larger program, answer these questions:

1. Are edges directed or undirected?
2. Are edges weighted or unweighted?
3. Can a vertex exist with no edges?
4. Should duplicate edges be ignored, rejected, or stored?
5. Which operations must be fast: edge lookup, neighbor iteration, adding edges, or removing edges?


```{rubric} Footnotes
```
[^1]: This lab intentionally stops at representation and simple queries. Traversal and shortest-path algorithms appear in later algorithm chapters.
[^2]: A graph can be stored many other ways, including edge lists and compressed sparse row structures. Adjacency matrices and lists are the best starting point for introductory C# work.
